# Bayesian HMM via PyMC (Phase A.6 + I.3)

Full Bayesian inference of every HMM parameter (transition matrix, startprob, emission means and scales) via NUTS sampling. This notebook demonstrates Phase A.6 (`BayesianHMMBackend`) and Phase I.3 (PyMC bridge) — they're the same thing : the Bayesian backend IS the PyMC bridge.

**Why this exists** : the frequentist Baum-Welch backend gives point estimates. For research applications where credible intervals on parameters matter (publication, regulatory compliance, model-uncertainty quantification), the Bayesian posterior is more informative.

**Scope (MVP) :**
- Gaussian diag-covariance emissions only
- Ergodic topologies only (no left-right / constrained mask)
- Slower than frequentist (NUTS ≈ 1 min for T=300, K=3) — research-grade, not production

**Install :** `pip install "hmm-studio[bayesian]"` — pymc is an optional dependency.

## 1. Generate synthetic 2-regime Gaussian data

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
X = np.concatenate([
    rng.normal(0.0, 0.5, (50, 1)),
    rng.normal(5.0, 0.5, (50, 1)),
    rng.normal(0.0, 0.5, (50, 1)),
    rng.normal(5.0, 0.5, (50, 1)),
])
X.shape

## 2. Declare an ergodic topology (same as frequentist)

The same `Topology` object works for frequentist or Bayesian backends — the choice is at fit time. Same YAML files, same Python builders.

In [ ]:
from hmm_core.topology import Topology, EmissionSpec, FitSpec, InitSpec

topo = Topology(
    name="bayes_demo",
    n_states=2,
    state_names=["a", "b"],
    emission=EmissionSpec(type="gaussian", covariance_type="diag", n_features=1),
    allowed_transitions=None,      # MVP supports ergodic only
    startprob="uniform",
    init=InitSpec(strategy="kmeans", seed=42),
    fit=FitSpec(algorithm="baum_welch", n_iter=10, tol=1e-3),
)
topo

## 3. Fit via Bayesian backend

Just pass `backend="bayesian"` to `fit()`. NUTS sampler runs ~1 min on small problems.

In [ ]:
from hmm_core.fit import fit
from hmm_core.backends import BayesianHMMBackend

# Tune sampler hyperparameters
backend = BayesianHMMBackend(
    n_samples=300,    # post-warmup draws per chain
    n_tune=200,       # NUTS warmup iterations
    n_chains=2,       # parallel MCMC chains
    target_accept=0.9,
)

result = fit(topo, X, seed=42, backend=backend)
result      # rich HTML view of posterior means

## 4. Inspect the posterior

The `last_idata_` attribute holds the full `arviz.InferenceData`. We can get credible intervals on every parameter.

In [ ]:
idata = backend.last_idata_
print("Posterior groups :", list(idata.posterior.data_vars))

# Posterior mean of emission means (per state, per feature)
mus_posterior = idata.posterior["mus"]
print("\nmus posterior summary :")
print(f"  shape (chain, draw, K, D) : {tuple(mus_posterior.shape)}")
print(f"  mean over (chain, draw)   :\n{mus_posterior.mean(('chain', 'draw')).values}")

In [ ]:
# 94% credible intervals on emission means
import arviz as az

summary = az.summary(idata, var_names=["mus", "sigmas", "transmat"])
summary

## 5. Use the posterior-mean model like any other FittedModel

The `result.model` is an hmmlearn `GaussianHMM` populated with posterior means. `predict`, `predict_proba`, `score` all work — the Bayesian backend is a drop-in replacement.

In [ ]:
viterbi = result.model.predict(X)
print("Decoded Viterbi (first 30) :", viterbi[:30])
print(f"\nLog-likelihood (under posterior mean) : {result.log_likelihood:.3f}")
print(f"BIC : {result.bic:.3f}")
print(f"AIC : {result.aic:.3f}")

## 6. When to use Bayesian vs frequentist ?

**Bayesian backend (this notebook)** :
- ✅ Credible intervals on every parameter
- ✅ Posterior predictive checks for model validation
- ✅ Natural model-uncertainty quantification
- ❌ Slow (NUTS sampling)
- ❌ MVP : Gaussian diag only, ergodic only

**Frequentist backend (`HmmlearnBackend`, default)** :
- ✅ Fast (Baum-Welch EM in C)
- ✅ All 4 emission types (Gaussian, GMM, Multinomial, Poisson)
- ✅ Constrained masks (left-right, lifecycle, etc.)
- ✅ NHMM, GMM-NHMM, Factorial NHMM extensions
- ❌ Point estimates only — no uncertainty quantification on params

**Hybrid workflow** (common in publication-grade research) :
1. Use the frequentist backend for model selection (BIC across K)
2. Pick the best K
3. Refit that K with the Bayesian backend to get credible intervals for the paper

## Strategic positioning

See [ADR-0012](../docs/decisions/0012-distribution-strategy-hybrid.md) — the Bayesian backend is Phase I.3 of the distribution strategy : we *bridge to* PyMC rather than *fork* it. The HMM math stays in `hmm_core` ; PyMC provides the inference engine for the Bayesian formulation.